# DOGE-Adapted Bezier Centerline Extraction

**HERE Cairo Hackathon 2026 — Problem 1: Centerline Generation**

This notebook adapts the [DOGE paper](https://arxiv.org/abs/2412.xxxxx) (Differentiable Optimization of Bezier Graph)
for road centerline extraction from GPS traces (VPD + HPD/Probe data).

### Pipeline Overview
1. **Data Loading** — Load VPD/HPD traces, clip to Kosovo bbox, project to UTM
2. **Rasterization** — Convert GPS traces into a density segmentation mask
3. **Tiling** — Split into overlapping 512x512 tiles
4. **DOGE Optimization** — Per-tile: alternate TopoAdapt (topology) + DiffAlign (geometry)
5. **Stitching** — Merge tile results, snap boundary nodes
6. **Export & Evaluate** — Output centerlines as GPKG, evaluate against ground truth

## 0. Setup & Dependencies

In [ ]:
import subprocess, sys

# Install core deps (most are pre-installed on Kaggle)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'geopandas', 'pyproj', 'shapely', 'scipy', 'torch', 'matplotlib'])

# Try installing DiffVG (optional — falls back to pure PyTorch if unavailable)
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'diffvg'])
    print('DiffVG installed successfully')
except Exception:
    print('DiffVG not available — using PyTorch soft renderer (works fine for demo)')

In [ ]:
import ast, json, os, time, warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from pyproj import CRS, Transformer
from scipy.ndimage import gaussian_filter, label as nd_label, maximum_filter
from scipy.spatial import cKDTree
from shapely import wkt
from shapely.geometry import LineString, MultiLineString, box
from shapely.ops import linemerge, transform

warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}, device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

## 1. Data Loading

Upload your data to Kaggle as a dataset, then set `DATA_DIR` below.
Expected structure:
```
DATA_DIR/
  Kosovo_VPD/Kosovo_VPD.parquet
  Kosovo_HPD/XKO_HPD_week_1.parquet
  Kosovo_HPD/XKO_HPD_week_2.parquet
  Kosovo's nav streets/nav_kosovo.parquet   (ground truth, optional)
```

In [ ]:
# === CONFIGURE THESE PATHS ===
DATA_DIR = Path('/kaggle/input/here-hackathon-data')  # Adjust to your Kaggle dataset path
OUTPUT_DIR = Path('/kaggle/working/bezier_doge_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Kosovo bounding box (WGS84)
BBOX = (21.088588, 42.571255, 21.188588, 42.671255)
UTM_EPSG = 32634  # UTM zone 34N
RESOLUTION_M = 1.0  # meters per pixel

In [ ]:
def parse_wkt_multi(text):
    """Parse WKT string into list of LineStrings."""
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return []
    try:
        geom = wkt.loads(str(text))
    except Exception:
        return []
    if geom is None or geom.is_empty:
        return []
    if isinstance(geom, LineString):
        return [geom]
    if isinstance(geom, MultiLineString):
        return [g for g in geom.geoms if isinstance(g, LineString) and not g.is_empty]
    return []


def clip_line(geom, clip_poly):
    """Clip LineString to polygon, return largest piece."""
    try:
        clipped = geom.intersection(clip_poly)
    except Exception:
        return None
    if clipped is None or clipped.is_empty:
        return None
    if isinstance(clipped, LineString):
        return clipped
    if isinstance(clipped, MultiLineString):
        parts = [g for g in clipped.geoms if isinstance(g, LineString) and not g.is_empty]
        return max(parts, key=lambda g: g.length) if parts else None
    return None


def load_vpd(path, bbox, fused_only=True):
    """Load VPD traces clipped to bbox."""
    suffix = path.suffix.lower()
    usecols = ['driveid', 'fused', 'path']
    df = pd.read_parquet(path, columns=usecols) if suffix == '.parquet' else pd.read_csv(path, usecols=usecols)
    if fused_only and 'fused' in df.columns:
        df = df[df['fused'].astype(str).str.lower().isin({'yes', 'true', '1'})]
    clip_poly = box(*bbox)
    lines = []
    for _, row in df.iterrows():
        for geom in parse_wkt_multi(row.get('path')):
            c = clip_line(geom, clip_poly)
            if c is not None and c.length > 0:
                lines.append(c)
    return lines


def load_hpd(paths, bbox):
    """Load HPD probe points, group by trace, form LineStrings."""
    usecols = ['heading', 'latitude', 'longitude', 'traceid', 'speed', 'day', 'time']
    frames = []
    for p in paths:
        suffix = p.suffix.lower()
        sub = pd.read_parquet(p, columns=usecols) if suffix == '.parquet' else pd.read_csv(p, usecols=usecols)
        sub['latitude'] = pd.to_numeric(sub['latitude'], errors='coerce')
        sub['longitude'] = pd.to_numeric(sub['longitude'], errors='coerce')
        sub['day'] = pd.to_numeric(sub['day'], errors='coerce')
        sub = sub.dropna(subset=['latitude', 'longitude', 'traceid', 'time'])
        frames.append(sub)
    if not frames:
        return []
    all_pts = pd.concat(frames, ignore_index=True)
    all_pts['time'] = pd.to_datetime(all_pts['time'], format='%H:%M:%S', errors='coerce')
    all_pts = all_pts.dropna(subset=['time'])
    clip_poly = box(*bbox)
    lines = []
    for _, grp in all_pts.sort_values(['traceid', 'day', 'time']).groupby(['traceid', 'day'], sort=False):
        if len(grp) < 2:
            continue
        coords = list(zip(grp['longitude'].tolist(), grp['latitude'].tolist()))
        try:
            geom = LineString(coords)
        except Exception:
            continue
        if geom.is_empty or geom.length == 0:
            continue
        c = clip_line(geom, clip_poly)
        if c is not None and c.length > 0:
            lines.append(c)
    return lines

In [ ]:
# Find data files
vpd_dir = DATA_DIR / 'Kosovo_VPD'
hpd_dir = DATA_DIR / 'Kosovo_HPD'

vpd_file = next((f for ext in ['.parquet', '.csv'] for f in vpd_dir.glob(f'*{ext}')), None)
hpd_files = sorted([f for ext in ['.parquet', '.csv'] for f in hpd_dir.glob(f'*{ext}')])

print(f'VPD file: {vpd_file}')
print(f'HPD files: {hpd_files}')

# Load traces
vpd_lines = load_vpd(vpd_file, BBOX) if vpd_file else []
hpd_lines = load_hpd(hpd_files, BBOX) if hpd_files else []
print(f'\nLoaded {len(vpd_lines)} VPD traces, {len(hpd_lines)} HPD traces')

# Project to UTM and clean
utm_crs = CRS.from_epsg(UTM_EPSG)
to_utm = Transformer.from_crs(CRS.from_epsg(4326), utm_crs, always_xy=True)

lines_utm, sources = [], []
for geom in vpd_lines:
    proj = transform(to_utm.transform, geom)
    if not proj.is_empty and proj.length >= 10:
        s = proj.simplify(1.5, preserve_topology=False)
        if isinstance(s, LineString) and not s.is_empty and len(s.coords) >= 2:
            proj = s
        lines_utm.append(proj)
        sources.append('VPD')

for geom in hpd_lines:
    proj = transform(to_utm.transform, geom)
    if not proj.is_empty and proj.length >= 10:
        s = proj.simplify(3.0, preserve_topology=False)
        if isinstance(s, LineString) and not s.is_empty and len(s.coords) >= 2:
            proj = s
        lines_utm.append(proj)
        sources.append('HPD')

n_vpd = sum(1 for s in sources if s == 'VPD')
n_hpd = sum(1 for s in sources if s == 'HPD')
print(f'After projection: {n_vpd} VPD, {n_hpd} HPD traces')

# Compute UTM bounds
all_coords = np.vstack([np.array(l.coords) for l in lines_utm])
utm_bounds = (all_coords[:,0].min(), all_coords[:,1].min(),
              all_coords[:,0].max(), all_coords[:,1].max())
print(f'UTM bounds: {utm_bounds}')

## 2. Rasterize Traces into Segmentation Mask

This is the key adaptation: instead of SAM2 segmentation of satellite imagery,
we rasterize GPS traces into a density heatmap that serves as our target mask `S`.

In [ ]:
# Rasterization config
VPD_BUFFER_M = 3.0
HPD_BUFFER_M = 5.0
VPD_WEIGHT = 2.0
HPD_WEIGHT = 1.0
GAUSSIAN_SIGMA = 2.0
MASK_THRESHOLD = 0.3
PADDING_M = 50.0

# Build raster bounds
minx = utm_bounds[0] - PADDING_M
miny = utm_bounds[1] - PADDING_M
maxx = utm_bounds[2] + PADDING_M
maxy = utm_bounds[3] + PADDING_M
W_px = int(np.ceil((maxx - minx) / RESOLUTION_M))
H_px = int(np.ceil((maxy - miny) / RESOLUTION_M))
print(f'Raster size: {W_px} x {H_px} pixels ({maxx-minx:.0f} x {maxy-miny:.0f} m)')

# Rasterize
density = np.zeros((H_px, W_px), dtype=np.float64)

for line, source in zip(lines_utm, sources):
    buffer_m = VPD_BUFFER_M if source == 'VPD' else HPD_BUFFER_M
    weight = VPD_WEIGHT if source == 'VPD' else HPD_WEIGHT

    total_len = line.length
    step = RESOLUTION_M * 0.5
    n_samples = max(int(total_len / step), 2)
    fracs = np.linspace(0.0, 1.0, n_samples)
    pts = np.array([(line.interpolate(f, normalized=True).x,
                     line.interpolate(f, normalized=True).y) for f in fracs])
    px = ((pts[:, 0] - minx) / RESOLUTION_M).astype(np.int32)
    py = ((maxy - pts[:, 1]) / RESOLUTION_M).astype(np.int32)
    valid = (px >= 0) & (px < W_px) & (py >= 0) & (py < H_px)
    np.add.at(density, (py[valid], px[valid]), weight)

# Dilate to approximate line buffering
avg_buffer_px = max(1, int(np.mean([VPD_BUFFER_M, HPD_BUFFER_M]) / RESOLUTION_M))
density = maximum_filter(density, size=2 * avg_buffer_px + 1)

# Normalize + smooth
if density.max() > 0:
    density /= density.max()
density = gaussian_filter(density, sigma=GAUSSIAN_SIGMA)
if density.max() > 0:
    density /= density.max()

soft_mask = density.astype(np.float32)
binary_mask = (soft_mask >= MASK_THRESHOLD).astype(np.float32)
RASTER_ORIGIN = (minx, maxy)  # top-left in UTM

coverage = binary_mask.sum() / binary_mask.size * 100
print(f'Mask coverage: {coverage:.1f}%')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
axes[0].imshow(soft_mask, cmap='hot', origin='upper')
axes[0].set_title('Soft Density Mask (target S)')
axes[1].imshow(binary_mask, cmap='gray', origin='upper')
axes[1].set_title(f'Binary Mask (threshold={MASK_THRESHOLD})')
for ax in axes:
    ax.set_axis_off()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'raster_mask.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Tiling

In [ ]:
TILE_SIZE = 512
TILE_OVERLAP = 64
MIN_TILE_COVERAGE = 0.02

@dataclass
class Tile:
    row: int; col: int
    y_start: int; y_end: int; x_start: int; x_end: int
    soft_mask: np.ndarray = None
    coverage: float = 0.0
    origin_utm_x: float = 0.0
    origin_utm_y: float = 0.0

step = TILE_SIZE - 2 * TILE_OVERLAP
n_rows = max(1, int(np.ceil((H_px - TILE_OVERLAP) / step)))
n_cols = max(1, int(np.ceil((W_px - TILE_OVERLAP) / step)))

tiles = []
for r in range(n_rows):
    for c in range(n_cols):
        y0 = r * step
        x0 = c * step
        y1 = min(y0 + TILE_SIZE, H_px)
        x1 = min(x0 + TILE_SIZE, W_px)
        if y1 - y0 < 64 or x1 - x0 < 64:
            continue
        tile_bin = binary_mask[y0:y1, x0:x1]
        cov = tile_bin.sum() / tile_bin.size
        if cov >= MIN_TILE_COVERAGE:
            tiles.append(Tile(
                row=r, col=c, y_start=y0, y_end=y1, x_start=x0, x_end=x1,
                soft_mask=soft_mask[y0:y1, x0:x1].copy(),
                coverage=cov,
                origin_utm_x=RASTER_ORIGIN[0] + x0 * RESOLUTION_M,
                origin_utm_y=RASTER_ORIGIN[1] - y0 * RESOLUTION_M,
            ))

print(f'Tile grid: {n_rows} x {n_cols} = {n_rows*n_cols} total, {len(tiles)} non-empty')

## 4. Bezier Graph Data Structure

The core representation from the DOGE paper (Section 3.1).
Each edge is a cubic Bezier curve with 4 control points,
reparameterized via alpha (chord projection) and d (perpendicular offset).

In [ ]:
class BezierGraph(nn.Module):
    """Parametric road network graph using cubic Bezier curves."""

    def __init__(self, canvas_h, canvas_w):
        super().__init__()
        self.canvas_h = canvas_h
        self.canvas_w = canvas_w
        self.node_positions = nn.Parameter(torch.zeros(0, 2))
        self.register_buffer('edge_indices', torch.zeros(0, 2, dtype=torch.long))
        self.edge_alpha = nn.Parameter(torch.zeros(0, 2))
        self.edge_offset = nn.Parameter(torch.zeros(0, 2))
        self.edge_width = nn.Parameter(torch.zeros(0))
        self._n_nodes = 0
        self._n_edges = 0

    @property
    def n_nodes(self): return self._n_nodes
    @property
    def n_edges(self): return self._n_edges

    def add_nodes(self, positions):
        K = positions.shape[0]
        start = self._n_nodes
        if self._n_nodes == 0:
            self.node_positions = nn.Parameter(positions.clone().float())
        else:
            self.node_positions = nn.Parameter(
                torch.cat([self.node_positions.data, positions.float()], 0))
        self._n_nodes += K
        return torch.arange(start, start + K, device=positions.device)

    def add_edges(self, start_nodes, end_nodes, alphas=None, offsets=None, widths=None):
        K = start_nodes.shape[0]
        dev = start_nodes.device
        if alphas is None:
            alphas = torch.zeros(K, 2, device=dev)
            alphas[:, 0] = 1/3; alphas[:, 1] = 2/3
        if offsets is None:
            offsets = torch.zeros(K, 2, device=dev)
        if widths is None:
            widths = torch.full((K,), 3.0, device=dev)
        new_idx = torch.stack([start_nodes.long(), end_nodes.long()], 1)
        if self._n_edges == 0:
            self.edge_indices = new_idx
            self.edge_alpha = nn.Parameter(alphas.clone().float())
            self.edge_offset = nn.Parameter(offsets.clone().float())
            self.edge_width = nn.Parameter(widths.clone().float())
        else:
            self.edge_indices = torch.cat([self.edge_indices, new_idx], 0)
            self.edge_alpha = nn.Parameter(torch.cat([self.edge_alpha.data, alphas.float()], 0))
            self.edge_offset = nn.Parameter(torch.cat([self.edge_offset.data, offsets.float()], 0))
            self.edge_width = nn.Parameter(torch.cat([self.edge_width.data, widths.float()], 0))
        self._n_edges += K
        return torch.arange(self._n_edges - K, self._n_edges, device=dev)

    def compute_control_points(self, edge_idx=None):
        """Eq 2a-2d: compute P0, P1, P2, P3 for each edge."""
        if edge_idx is None:
            edge_idx = torch.arange(self._n_edges, device=self.node_positions.device)
        if len(edge_idx) == 0:
            return torch.zeros(0, 4, 2, device=self.node_positions.device)
        idx = self.edge_indices[edge_idx]
        P0 = self.node_positions[idx[:, 0]]
        P3 = self.node_positions[idx[:, 1]]
        alpha = torch.clamp(self.edge_alpha[edge_idx], 0, 1)
        offset = self.edge_offset[edge_idx]
        chord = P3 - P0
        chord_len = torch.norm(chord, dim=1, keepdim=True).clamp(min=1e-6)
        chord_dir = chord / chord_len
        normal = torch.stack([-chord_dir[:, 1], chord_dir[:, 0]], dim=1)
        P1 = (1 - alpha[:, 0:1]) * P0 + alpha[:, 0:1] * P3 + offset[:, 0:1] * normal
        P2 = (1 - alpha[:, 1:2]) * P0 + alpha[:, 1:2] * P3 + offset[:, 1:2] * normal
        return torch.stack([P0, P1, P2, P3], dim=1)

    def sample_curves(self, n_samples=50, edge_idx=None):
        """Eq 1: evaluate cubic Bezier at uniform t values."""
        cp = self.compute_control_points(edge_idx)
        if cp.shape[0] == 0:
            return torch.zeros(0, n_samples, 2, device=cp.device)
        t = torch.linspace(0, 1, n_samples, device=cp.device).unsqueeze(0).unsqueeze(-1)
        P0, P1, P2, P3 = cp[:, 0:1], cp[:, 1:2], cp[:, 2:3], cp[:, 3:4]
        omt = 1.0 - t
        return omt**3*P0 + 3*omt**2*t*P1 + 3*omt*t**2*P2 + t**3*P3

    def get_edge_tangents(self, edge_idx=None):
        cp = self.compute_control_points(edge_idx)
        if cp.shape[0] == 0:
            e = torch.zeros(0, 2, device=cp.device)
            return e, e
        return 3*(cp[:,1]-cp[:,0]), 3*(cp[:,3]-cp[:,2])

    def node_degree(self):
        deg = torch.zeros(self._n_nodes, dtype=torch.long, device=self.node_positions.device)
        if self._n_edges > 0:
            for i in range(2):
                deg.scatter_add_(0, self.edge_indices[:, i],
                                 torch.ones(self._n_edges, dtype=torch.long, device=deg.device))
        return deg

    def edges_at_node(self, node_idx):
        result = []
        for e in range(self._n_edges):
            if self.edge_indices[e, 0].item() == node_idx: result.append((e, True))
            if self.edge_indices[e, 1].item() == node_idx: result.append((e, False))
        return result

    def remove_edges(self, mask):
        keep = ~mask
        if keep.all(): return
        self.edge_indices = self.edge_indices[keep]
        self.edge_alpha = nn.Parameter(self.edge_alpha.data[keep])
        self.edge_offset = nn.Parameter(self.edge_offset.data[keep])
        self.edge_width = nn.Parameter(self.edge_width.data[keep])
        self._n_edges = int(keep.sum().item())

    def remove_isolated_nodes(self):
        if self._n_nodes == 0: return
        deg = self.node_degree()
        keep = deg > 0
        if keep.all(): return
        new_idx = torch.full((self._n_nodes,), -1, dtype=torch.long, device=keep.device)
        new_idx[keep] = torch.arange(keep.sum().item(), device=keep.device)
        self.node_positions = nn.Parameter(self.node_positions.data[keep])
        self._n_nodes = int(keep.sum().item())
        if self._n_edges > 0:
            self.edge_indices = new_idx[self.edge_indices]

    def to_polylines(self, samples=50):
        with torch.no_grad():
            if self._n_edges == 0: return []
            pts = self.sample_curves(n_samples=samples)
            return [pts[e].cpu().numpy() for e in range(pts.shape[0])]

    def get_optimizable_params(self):
        return [self.node_positions, self.edge_alpha, self.edge_offset, self.edge_width]

    def __repr__(self):
        return f'BezierGraph(nodes={self._n_nodes}, edges={self._n_edges}, canvas={self.canvas_h}x{self.canvas_w})'

## 5. Differentiable Renderer

Soft Gaussian renderer in pure PyTorch: sample points along each Bezier curve,
apply a Gaussian kernel to compute each pixel's coverage. Fully differentiable.

In [ ]:
def render_graph(graph, canvas_h=None, canvas_w=None, n_samples=64):
    """Differentiable soft rendering of a BezierGraph."""
    H = canvas_h or graph.canvas_h
    W = canvas_w or graph.canvas_w
    dev = graph.node_positions.device
    if graph.n_edges == 0:
        return torch.zeros(H, W, device=dev)

    curve_pts = graph.sample_curves(n_samples=n_samples)  # (E, T, 2)
    widths = torch.abs(graph.edge_width).clamp(min=0.5)
    canvas = torch.zeros(H, W, device=dev)

    for b in range(curve_pts.shape[0]):
        pts = curve_pts[b]  # (T, 2)
        sig = widths[b]
        pad = sig * 3
        x_min = max(0, int(pts[:, 0].min().item() - pad.item()))
        x_max = min(W, int(pts[:, 0].max().item() + pad.item()) + 1)
        y_min = max(0, int(pts[:, 1].min().item() - pad.item()))
        y_max = min(H, int(pts[:, 1].max().item() + pad.item()) + 1)
        if x_max <= x_min or y_max <= y_min:
            continue
        gy, gx = torch.meshgrid(
            torch.arange(y_min, y_max, device=dev, dtype=torch.float32),
            torch.arange(x_min, x_max, device=dev, dtype=torch.float32),
            indexing='ij')
        pixels = torch.stack([gx, gy], -1).reshape(-1, 1, 2)  # (P, 1, 2)
        dist_sq = ((pixels - pts.unsqueeze(0))**2).sum(-1).min(1).values  # (P,)
        contribution = torch.exp(-dist_sq / (2 * sig * sig))
        canvas[y_min:y_max, x_min:x_max] = torch.max(
            canvas[y_min:y_max, x_min:x_max], contribution.reshape(y_max-y_min, x_max-x_min))

    return canvas.clamp(0, 1)


def render_single_edge(graph, edge_idx, H, W, n_samples=64):
    """Render a single edge for overlap computation."""
    dev = graph.node_positions.device
    idx = torch.tensor([edge_idx], device=dev)
    pts = graph.sample_curves(n_samples=n_samples, edge_idx=idx)[0]
    sig = torch.abs(graph.edge_width[edge_idx]).clamp(min=0.5)
    canvas = torch.zeros(H, W, device=dev)
    pad = sig * 3
    x_min = max(0, int(pts[:, 0].min().item() - pad.item()))
    x_max = min(W, int(pts[:, 0].max().item() + pad.item()) + 1)
    y_min = max(0, int(pts[:, 1].min().item() - pad.item()))
    y_max = min(H, int(pts[:, 1].max().item() + pad.item()) + 1)
    if x_max <= x_min or y_max <= y_min:
        return canvas
    gy, gx = torch.meshgrid(
        torch.arange(y_min, y_max, device=dev, dtype=torch.float32),
        torch.arange(x_min, x_max, device=dev, dtype=torch.float32), indexing='ij')
    pixels = torch.stack([gx, gy], -1).reshape(-1, 1, 2)
    dist_sq = ((pixels - pts.unsqueeze(0))**2).sum(-1).min(1).values
    canvas[y_min:y_max, x_min:x_max] = torch.exp(-dist_sq / (2*sig*sig)).reshape(y_max-y_min, x_max-x_min)
    return canvas.clamp(0, 1)

## 6. DiffAlign — Loss Functions (Eq 3-8)

Five loss terms from the DOGE paper that jointly optimize geometry.

In [ ]:
# Loss weights from paper (Section 4.1)
LAMBDA_COVER = 1.0
LAMBDA_OVERLAP = 0.3
LAMBDA_G1 = 0.012
LAMBDA_OFFSET = 0.006
LAMBDA_SPACING = 0.006
N_SAMPLES = 48

def loss_coverage(rendered, target):
    """Eq 4: pixel-wise L2."""
    return F.mse_loss(rendered, target)

def loss_overlap(graph, H, W, max_edges=80):
    """Eq 5: penalize multi-covered pixels."""
    N = graph.n_edges
    if N <= 1: return torch.tensor(0.0, device=graph.node_positions.device)
    edge_list = list(range(min(N, max_edges)))
    s = torch.zeros(H, W, device=graph.node_positions.device)
    for e in edge_list:
        s = s + render_single_edge(graph, e, H, W, n_samples=32)
    return torch.clamp(s - 1.0, min=0).mean()

def loss_g1(graph, threshold_deg=160.0):
    """Eq 6: tangent alignment at degree-2 nodes."""
    if graph.n_nodes == 0 or graph.n_edges < 2:
        return torch.tensor(0.0, device=graph.node_positions.device)
    deg = graph.node_degree()
    deg2 = torch.where(deg == 2)[0]
    if len(deg2) == 0:
        return torch.tensor(0.0, device=graph.node_positions.device)
    st, et = graph.get_edge_tangents()
    threshold_rad = threshold_deg * np.pi / 180.0
    loss_sum = torch.tensor(0.0, device=graph.node_positions.device)
    count = 0
    for ni in deg2.tolist():
        edges = graph.edges_at_node(ni)
        if len(edges) != 2: continue
        e1, s1 = edges[0]; e2, s2 = edges[1]
        t1 = st[e1] if s1 else -et[e1]
        t2 = st[e2] if s2 else -et[e2]
        t1n = t1 / (torch.norm(t1) + 1e-8)
        t2n = t2 / (torch.norm(t2) + 1e-8)
        cos_a = torch.dot(t1n, t2n).clamp(-1, 1)
        angle = torch.acos(cos_a)
        if angle.item() < threshold_rad:
            loss_sum = loss_sum + (1.0 - cos_a)
            count += 1
    return loss_sum / max(count, 1)

def loss_offset(graph, tau=0.5):
    """Eq 7: penalize excessive control point offsets."""
    if graph.n_edges == 0:
        return torch.tensor(0.0, device=graph.node_positions.device)
    idx = graph.edge_indices
    L = torch.norm(graph.node_positions[idx[:,1]] - graph.node_positions[idx[:,0]], dim=1).clamp(min=1e-6)
    ratios = torch.abs(graph.edge_offset) / L.unsqueeze(1)
    return torch.clamp(torch.exp(ratios - tau) - 1.0, min=0).mean()

def loss_spacing(graph):
    """Eq 8: encourage equidistant control point placement."""
    if graph.n_edges == 0:
        return torch.tensor(0.0, device=graph.node_positions.device)
    a = graph.edge_alpha
    return ((a[:, 0] - 1/3)**2 + (a[:, 1] - 2/3)**2).mean()

def compute_losses(graph, target, rendered):
    """Eq 3: composite loss."""
    H, W = target.shape
    l_c = loss_coverage(rendered, target)
    l_o = loss_overlap(graph, H, W) if graph.n_edges > 1 else torch.tensor(0.0, device=target.device)
    l_g = loss_g1(graph)
    l_off = loss_offset(graph)
    l_sp = loss_spacing(graph)
    total = (LAMBDA_COVER*l_c + LAMBDA_OVERLAP*l_o + LAMBDA_G1*l_g
             + LAMBDA_OFFSET*l_off + LAMBDA_SPACING*l_sp)
    return total, {'total': total.item(), 'cover': l_c.item(), 'overlap': l_o.item(),
                   'g1': l_g.item(), 'offset': l_off.item(), 'spacing': l_sp.item()}

## 7. TopoAdapt — Topology Operators (Section 3.3)

Five discrete operators that refine the graph's connectivity between gradient steps.

In [ ]:
MERGE_DIST = 4.0  # pixels (~meters at 1m/px)
NEW_ROAD_LEN = 40.0
NEW_ROAD_WIDTH = 3.0
MAX_NEW_ROADS = 10
MIN_EDGE_LEN = 5.0
COLLINEAR_THRESH_DEG = 160.0


def op_road_addition(graph, target, max_new=MAX_NEW_ROADS):
    """Eq 9-10: seed edges in uncovered road regions."""
    H, W = target.shape
    dev = graph.node_positions.device
    with torch.no_grad():
        rendered = render_graph(graph, H, W, n_samples=32).cpu().numpy()
    target_np = target.detach().cpu().numpy()
    unfit = ((target_np > 0.5) & (rendered < 0.3)).astype(np.float32)
    labeled, n_comp = nd_label(unfit)
    if n_comp == 0: return 0
    centers, sizes = [], []
    for c in range(1, n_comp + 1):
        m = labeled == c
        sz = m.sum()
        if sz < 10: continue
        ys, xs = np.where(m)
        centers.append((xs.mean(), ys.mean()))
        sizes.append(sz)
    if not centers: return 0
    order = np.argsort(sizes)[::-1][:max_new]
    added = 0
    for i in order:
        cx, cy = centers[i]
        if graph.n_nodes > 0:
            pos_np = graph.node_positions.detach().cpu().numpy()
            if np.sqrt((pos_np[:,0]-cx)**2+(pos_np[:,1]-cy)**2).min() < MERGE_DIST*2:
                continue
        angle = np.random.uniform(0, np.pi)
        hl = NEW_ROAD_LEN / 2
        x0 = float(np.clip(cx - hl*np.cos(angle), 1, W-2))
        y0 = float(np.clip(cy - hl*np.sin(angle), 1, H-2))
        x1 = float(np.clip(cx + hl*np.cos(angle), 1, W-2))
        y1 = float(np.clip(cy + hl*np.sin(angle), 1, H-2))
        nids = graph.add_nodes(torch.tensor([[x0,y0],[x1,y1]], device=dev))
        graph.add_edges(nids[0:1], nids[1:2], widths=torch.tensor([NEW_ROAD_WIDTH], device=dev))
        added += 1
    return added


def op_node_merging(graph):
    """Merge nodes closer than MERGE_DIST."""
    if graph.n_nodes < 2: return 0
    pos = graph.node_positions.detach().cpu().numpy()
    tree = cKDTree(pos)
    pairs = tree.query_pairs(r=MERGE_DIST)
    if not pairs: return 0
    parent = list(range(graph.n_nodes))
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for i, j in pairs: ri, rj = find(i), find(j); parent[rj] = ri if ri != rj else parent[rj]
    canonical = {i: find(i) for i in range(graph.n_nodes)}
    if len(set(canonical.values())) == graph.n_nodes: return 0
    root_pos = {}
    for i in range(graph.n_nodes):
        r = canonical[i]
        root_pos.setdefault(r, []).append(pos[i])
    r2new = {}; new_pos = []
    for r in sorted(root_pos): r2new[r] = len(new_pos); new_pos.append(np.mean(root_pos[r], axis=0))
    o2n = {i: r2new[canonical[i]] for i in range(graph.n_nodes)}
    dev = graph.node_positions.device
    new_pos_t = torch.tensor(np.array(new_pos, dtype=np.float32), device=dev)
    if graph.n_edges > 0:
        old_e = graph.edge_indices.cpu().numpy()
        new_e = np.array([[o2n[int(u)], o2n[int(v)]] for u,v in old_e], dtype=np.int64)
        valid = new_e[:,0] != new_e[:,1]
        graph.edge_indices = torch.tensor(new_e[valid], device=dev, dtype=torch.long)
        vt = torch.tensor(valid, device=dev, dtype=torch.bool)
        graph.edge_alpha = nn.Parameter(graph.edge_alpha.data[vt])
        graph.edge_offset = nn.Parameter(graph.edge_offset.data[vt])
        graph.edge_width = nn.Parameter(graph.edge_width.data[vt])
        graph._n_edges = int(valid.sum())
    graph.node_positions = nn.Parameter(new_pos_t)
    graph._n_nodes = len(new_pos)
    return len(pairs)


def op_t_junction(graph, n_samples=32):
    """Snap degree-1 nodes to nearby edges."""
    if graph.n_nodes < 1 or graph.n_edges < 1: return 0
    pos = graph.node_positions.detach().cpu().numpy()
    deg = graph.node_degree().cpu().numpy()
    candidates = np.where(deg == 1)[0]
    if len(candidates) == 0: return 0
    with torch.no_grad():
        all_pts = graph.sample_curves(n_samples=n_samples).cpu().numpy()
    created = 0
    for ni in candidates:
        np_ = pos[ni]
        connected = set(e for e, _ in graph.edges_at_node(ni))
        best_d, best_e = float('inf'), -1
        for e in range(graph.n_edges):
            if e in connected: continue
            d = np.sqrt(np.sum((all_pts[e] - np_)**2, axis=1)).min()
            if d < best_d: best_d, best_e = d, e
        if best_e < 0 or best_d > MERGE_DIST: continue
        es = int(graph.edge_indices[best_e, 0].item())
        ee = int(graph.edge_indices[best_e, 1].item())
        ds = np.linalg.norm(pos[es] - np_)
        de = np.linalg.norm(pos[ee] - np_)
        tgt = es if ds <= de else ee
        if tgt == ni: continue
        exists = any((graph.edge_indices[e,0].item()==ni and graph.edge_indices[e,1].item()==tgt) or
                     (graph.edge_indices[e,0].item()==tgt and graph.edge_indices[e,1].item()==ni)
                     for e in range(graph.n_edges))
        if exists: continue
        dev = graph.node_positions.device
        graph.add_edges(torch.tensor([ni],device=dev), torch.tensor([tgt],device=dev),
                        widths=torch.tensor([graph.edge_width[best_e].item()],device=dev))
        created += 1
    return created


def op_edge_pruning(graph):
    """Remove too-short or too-thin edges."""
    if graph.n_edges == 0: return 0
    with torch.no_grad():
        idx = graph.edge_indices
        chord = torch.norm(graph.node_positions[idx[:,1]] - graph.node_positions[idx[:,0]], dim=1)
        remove = (chord < MIN_EDGE_LEN) | (graph.edge_width < 1.0)
    n = int(remove.sum().item())
    if n > 0:
        graph.remove_edges(remove)
        graph.remove_isolated_nodes()
    return n


def topoadapt_step(graph, target, iteration):
    """Run all topology operators."""
    stats = {}
    stats['pruned'] = op_edge_pruning(graph)
    stats['merged'] = op_node_merging(graph)
    if iteration < 200 and iteration % 5 == 0:
        stats['added'] = op_road_addition(graph, target)
    else:
        stats['added'] = 0
    if iteration >= 10 and iteration % 10 == 0:
        stats['t_junc'] = op_t_junction(graph)
    else:
        stats['t_junc'] = 0
    return stats

## 8. DOGE Optimizer — Algorithm 1

The main loop: alternate TopoAdapt (topology) and DiffAlign (geometry).

In [ ]:
T_MAX = 120           # Max iterations per tile
LR = 0.5              # Learning rate
LR_DECAY = 0.995
DIFFALIGN_STEPS = 5   # Gradient steps per topo step
TOPO_INTERVAL = 10    # TopoAdapt every N iters
PATIENCE = 25
MAX_INITIAL_SEEDS = 25
LOG_INTERVAL = 20


def initialize_graph(target_t):
    """Seed initial edges in high-density regions."""
    H, W = target_t.shape
    dev = target_t.device
    graph = BezierGraph(H, W).to(dev)
    target_np = target_t.cpu().numpy()
    unfit = (target_np > 0.3).astype(np.float32)
    labeled, n_comp = nd_label(unfit)
    centers, sizes = [], []
    for c in range(1, n_comp + 1):
        m = labeled == c
        sz = m.sum()
        if sz < 10: continue
        ys, xs = np.where(m)
        centers.append((xs.mean(), ys.mean()))
        sizes.append(sz)
    order = np.argsort(sizes)[::-1][:MAX_INITIAL_SEEDS]
    for i in order:
        cx, cy = centers[i]
        angle = np.random.uniform(0, np.pi)
        hl = NEW_ROAD_LEN / 2
        x0 = float(np.clip(cx - hl*np.cos(angle), 1, W-2))
        y0 = float(np.clip(cy - hl*np.sin(angle), 1, H-2))
        x1 = float(np.clip(cx + hl*np.cos(angle), 1, W-2))
        y1 = float(np.clip(cy + hl*np.sin(angle), 1, H-2))
        nids = graph.add_nodes(torch.tensor([[x0,y0],[x1,y1]], device=dev))
        graph.add_edges(nids[0:1], nids[1:2], widths=torch.tensor([NEW_ROAD_WIDTH], device=dev))
    print(f'  [init] Seeded {graph.n_edges} edges')
    return graph


def optimize_tile(target_np):
    """Full DOGE optimization loop for one tile (Algorithm 1)."""
    dev = torch.device(DEVICE)
    target_t = torch.tensor(target_np, dtype=torch.float32, device=dev)
    H, W = target_t.shape

    graph = initialize_graph(target_t)
    if graph.n_edges == 0:
        return graph, []

    history = []
    best_loss, patience_ctr = float('inf'), 0

    for t in range(T_MAX):
        # TopoAdapt phase
        if t % TOPO_INTERVAL == 0:
            with torch.no_grad():
                stats = topoadapt_step(graph, target_t, t)
            if graph.n_edges == 0:
                graph = initialize_graph(target_t)
                if graph.n_edges == 0: break

        # Rebuild optimizer after topo changes
        if t % TOPO_INTERVAL == 0 or t == 0:
            lr = LR * (LR_DECAY ** t)
            optimizer = torch.optim.Adam(graph.get_optimizable_params(), lr=lr)

        # DiffAlign phase
        for _ in range(DIFFALIGN_STEPS):
            optimizer.zero_grad()
            try:
                rendered = render_graph(graph, H, W, n_samples=N_SAMPLES)
                total_loss, ld = compute_losses(graph, target_t, rendered)
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(graph.parameters(), 10.0)
                optimizer.step()
            except RuntimeError as e:
                if 'out of memory' in str(e).lower():
                    torch.cuda.empty_cache()
                    break
                raise
            with torch.no_grad():
                graph.edge_alpha.data.clamp_(0, 1)
                graph.edge_width.data.clamp_(min=0.5)
                graph.node_positions.data[:, 0].clamp_(0, W-1)
                graph.node_positions.data[:, 1].clamp_(0, H-1)

        history.append({'iter': t, **ld, 'nodes': graph.n_nodes, 'edges': graph.n_edges})

        if t % LOG_INTERVAL == 0:
            print(f'    iter {t:3d} | loss={ld["total"]:.4f} (cover={ld["cover"]:.4f}) | '
                  f'{graph.n_nodes}N {graph.n_edges}E')

        # Early stopping
        if ld['total'] < best_loss - 1e-5:
            best_loss = ld['total']; patience_ctr = 0
        else:
            patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'    Early stop at iter {t}')
            break

    print(f'  Final: {graph}')
    return graph, history

## 9. Run Optimization on All Tiles

In [ ]:
# Optional: limit tiles for quick testing
MAX_TILES = None  # Set to e.g. 5 for quick test, None for full run

run_tiles = tiles[:MAX_TILES] if MAX_TILES else tiles
print(f'Optimizing {len(run_tiles)} tiles...')

tile_results = []
all_history = []
t0 = time.time()

for i, tile in enumerate(run_tiles):
    print(f'\n=== Tile {i+1}/{len(run_tiles)} (r={tile.row}, c={tile.col}, cov={tile.coverage:.1%}) ===')
    try:
        graph, hist = optimize_tile(tile.soft_mask)
        tile_results.append((tile, graph))
        all_history.extend(hist)
    except Exception as e:
        print(f'  ERROR: {e}')
        import traceback; traceback.print_exc()

elapsed = time.time() - t0
print(f'\nDone! {len(tile_results)} tiles optimized in {elapsed:.1f}s')

## 10. Stitch Tiles & Export Results

In [ ]:
# Convert tile-local pixel coords to global UTM coords
all_polylines_utm = []
for tile, graph in tile_results:
    if graph.n_edges == 0: continue
    for poly_px in graph.to_polylines(samples=50):
        utm_x = tile.origin_utm_x + poly_px[:, 0] * RESOLUTION_M
        utm_y = tile.origin_utm_y - poly_px[:, 1] * RESOLUTION_M
        all_polylines_utm.append(np.column_stack([utm_x, utm_y]))

print(f'Total polylines before stitching: {len(all_polylines_utm)}')

# Merge boundary endpoints
STITCH_DIST = 4.0
if len(all_polylines_utm) > 1:
    endpoints = []
    for i, p in enumerate(all_polylines_utm):
        endpoints.append((i, 0, p[0]))
        endpoints.append((i, -1, p[-1]))
    ep_coords = np.array([e[2] for e in endpoints])
    tree = cKDTree(ep_coords)
    for i, j in tree.query_pairs(r=STITCH_DIST):
        pi, pp, pc = endpoints[i]; pj, jp, jc = endpoints[j]
        if pi == pj: continue
        mid = (pc + jc) / 2
        all_polylines_utm[pi][pp] = mid
        all_polylines_utm[pj][jp] = mid

print(f'Total polylines after stitching: {len(all_polylines_utm)}')

# Convert to WGS84 and save
import geopandas as gpd
to_wgs = Transformer.from_crs(utm_crs, CRS.from_epsg(4326), always_xy=True)

geometries = []
for poly_utm in all_polylines_utm:
    if len(poly_utm) < 2: continue
    line_utm = LineString(poly_utm)
    line_wgs = transform(to_wgs.transform, line_utm)
    if not line_wgs.is_empty:
        geometries.append(line_wgs)

print(f'Valid geometries: {len(geometries)}')

if geometries:
    gdf = gpd.GeoDataFrame(
        {'id': range(len(geometries)), 'source': ['bezier_doge'] * len(geometries)},
        geometry=geometries, crs='EPSG:4326')
    gpkg_path = OUTPUT_DIR / 'bezier_centerlines.gpkg'
    gdf.to_file(gpkg_path, driver='GPKG', layer='centerlines')
    print(f'Saved: {gpkg_path}')

    csv_path = OUTPUT_DIR / 'bezier_centerlines.csv'
    csv_df = gdf.copy()
    csv_df['geometry_wkt'] = csv_df.geometry.apply(lambda g: g.wkt)
    csv_df.drop(columns=['geometry']).to_csv(csv_path, index=False)
    print(f'Saved: {csv_path}')

## 11. Visualization

In [ ]:
# Plot the extracted centerlines on the density mask
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Left: density mask with centerlines overlaid
axes[0].imshow(soft_mask, cmap='gray', origin='upper',
               extent=[minx, maxx, miny, maxy])
for poly in all_polylines_utm:
    if len(poly) >= 2:
        axes[0].plot(poly[:, 0], poly[:, 1], 'r-', linewidth=0.5, alpha=0.7)
axes[0].set_title('Bezier Centerlines on Density Mask')
axes[0].set_aspect('equal')

# Right: centerlines only
for poly in all_polylines_utm:
    if len(poly) >= 2:
        axes[1].plot(poly[:, 0], poly[:, 1], 'b-', linewidth=0.8, alpha=0.8)
axes[1].set_title(f'Extracted Road Network ({len(all_polylines_utm)} segments)')
axes[1].set_aspect('equal')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'centerlines_overlay.png', dpi=150, bbox_inches='tight')
plt.show()

# Loss curve
if all_history:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot([h['total'] for h in all_history], alpha=0.3, label='Total')
    ax.plot([h['cover'] for h in all_history], alpha=0.7, label='Coverage')
    # Smoothed
    window = min(50, len(all_history)//4) or 1
    from numpy import convolve
    kernel = np.ones(window)/window
    ax.plot(convolve([h['total'] for h in all_history], kernel, mode='valid'),
            'k-', linewidth=2, label=f'Total (smoothed {window})')
    ax.set_xlabel('Step'); ax.set_ylabel('Loss'); ax.set_title('Optimization Loss')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.savefig(OUTPUT_DIR / 'loss_curve.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Visualize a few individual tile results
n_show = min(6, len(tile_results))
fig, axes = plt.subplots(2, n_show, figsize=(4*n_show, 8))
if n_show == 1:
    axes = axes.reshape(2, 1)

for i in range(n_show):
    tile, graph = tile_results[i]
    # Top: target mask
    axes[0, i].imshow(tile.soft_mask, cmap='gray', origin='upper')
    axes[0, i].set_title(f'Tile ({tile.row},{tile.col}) target')
    axes[0, i].set_axis_off()

    # Bottom: rendered graph
    if graph.n_edges > 0:
        with torch.no_grad():
            dev = torch.device(DEVICE)
            graph_dev = graph.to(dev)
            H, W = tile.soft_mask.shape
            rendered = render_graph(graph_dev, H, W, n_samples=64).cpu().numpy()
        axes[1, i].imshow(rendered, cmap='hot', origin='upper')
    axes[1, i].set_title(f'{graph.n_nodes}N {graph.n_edges}E')
    axes[1, i].set_axis_off()

plt.suptitle('Per-Tile: Target (top) vs Rendered Bezier Graph (bottom)', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'tile_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Evaluation (if ground truth available)

In [ ]:
# Look for ground truth
gt_candidates = [
    DATA_DIR / "Kosovo's nav streets" / 'nav_kosovo.parquet',
    DATA_DIR / "Kosovo's nav streets" / 'nav_kosovo.csv',
    DATA_DIR / 'nav_kosovo.parquet',
    DATA_DIR / 'nav_kosovo.csv',
]
gt_path = next((p for p in gt_candidates if p.exists()), None)

if gt_path and geometries:
    print(f'Ground truth found: {gt_path}')

    # Load ground truth
    gt_df = pd.read_parquet(gt_path) if gt_path.suffix == '.parquet' else pd.read_csv(gt_path)
    gt_geoms = []
    for s in gt_df['geom'].astype(str):
        try:
            g = wkt.loads(s)
            if g and not g.is_empty:
                gt_geoms.append(g)
        except: pass

    gt_gdf = gpd.GeoDataFrame({'id': range(len(gt_geoms))}, geometry=gt_geoms, crs='EPSG:4326')
    gen_gdf = gdf

    # Project both to UTM for metric computation
    gt_proj = gt_gdf.to_crs(f'EPSG:{UTM_EPSG}')
    gen_proj = gen_gdf.to_crs(f'EPSG:{UTM_EPSG}')

    # Length-based precision/recall (15m buffer)
    from shapely.ops import unary_union
    BUFFER_M = 15.0
    gt_buf = unary_union(gt_proj.buffer(BUFFER_M).tolist())
    gen_buf = unary_union(gen_proj.buffer(BUFFER_M).tolist())

    gen_len = float(gen_proj.length.sum())
    gt_len = float(gt_proj.length.sum())
    matched_gen = float(gen_proj.geometry.intersection(gt_buf).length.sum())
    matched_gt = float(gt_proj.geometry.intersection(gen_buf).length.sum())

    precision = matched_gen / gen_len if gen_len > 0 else 0
    recall = matched_gt / gt_len if gt_len > 0 else 0
    f1 = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0

    print(f'\n=== Evaluation Results ===')
    print(f'Generated: {len(gen_proj)} segments, {gen_len:.0f}m total')
    print(f'Ground truth: {len(gt_proj)} segments, {gt_len:.0f}m total')
    print(f'Precision: {precision:.4f}')
    print(f'Recall:    {recall:.4f}')
    print(f'F1:        {f1:.4f}')

    metrics = {'precision': precision, 'recall': recall, 'f1': f1,
               'gen_length_m': gen_len, 'gt_length_m': gt_len,
               'gen_count': len(gen_proj), 'gt_count': len(gt_proj)}
    with open(OUTPUT_DIR / 'metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2)

    # Overlay plot
    fig, ax = plt.subplots(figsize=(12, 12))
    gt_gdf.plot(ax=ax, linewidth=1.0, color='#1f77b4', alpha=0.6, label='Ground Truth')
    gen_gdf.plot(ax=ax, linewidth=1.0, color='#d62728', alpha=0.6, label='Bezier DOGE')
    ax.set_title(f'Centerline Comparison (P={precision:.3f} R={recall:.3f} F1={f1:.3f})')
    ax.legend(loc='lower left')
    ax.set_axis_off()
    plt.savefig(OUTPUT_DIR / 'gt_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Ground truth not found or no geometries generated. Skipping evaluation.')

In [ ]:
# Summary
print('\n' + '='*60)
print('DOGE-Adapted Bezier Centerline Extraction - Complete')
print('='*60)
print(f'Output directory: {OUTPUT_DIR}')
print(f'Files:')
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {f.name} ({f.stat().st_size/1024:.1f} KB)')